# Clean Data(1)

Legacy four-source workflow. Optional historical reproduction only; outputs are isolated in Legacy. The checkpoint uses row-order assumptions; verify the input order before reuse.

**Execution:** Start Jupyter inside the project tree. Run the path setup first. Review configuration and existing outputs before executing.

In [ ]:
# Resolve the project from the notebook working directory, not a machine-specific path.
from pathlib import Path
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if all((p / n).is_dir() for n in ("Python Code", "Data", "Streamlit"))), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Launch Jupyter from inside the AEGEAN project tree.")
SCRAPING_DIR = PROJECT_ROOT / "Data" / "Scraping"
CLEANING_DIR = PROJECT_ROOT / "Data" / "Output" / "Cleaning data"
HF_DIR = PROJECT_ROOT / "Data" / "Output" / "Hugging Face"
LLM_OUTPUT_DIR = PROJECT_ROOT / "Data" / "Output" / "LLM"
for directory in (CLEANING_DIR, HF_DIR, LLM_OUTPUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

LEGACY_DIR = CLEANING_DIR / "Legacy"
LEGACY_DIR.mkdir(exist_ok=True)


In [ ]:
from pathlib import Path

current_path = Path.cwd()
print("Current directory:", current_path)

In [ ]:
parent_path = current_path.parent
print("Parent directory:", parent_path)

In [ ]:
import pandas as pd

data_path = SCRAPING_DIR
airline_ratings = data_path  / "airlineratings_data/all_airlines_reviews.csv"

df_ar = pd.read_csv(airline_ratings)
df_ar.head()

In [ ]:
flight_report = data_path  / "flight_report_data/all_airlines_reviews.csv"

df_fr = pd.read_csv(flight_report)
df_fr.head()

In [ ]:
esky = data_path  / "esky/esky_all_reviews_v2.csv"

df_esky = pd.read_csv(esky)
df_esky.head()

In [ ]:
kayak = data_path  / "kayak/kayak_all_airline_reviews.csv"

df_kayak = pd.read_csv(kayak)
df_kayak.head()

In [ ]:
#Select specific Airline Ratings Columns
df = df_ar[["airline", "review_date", "rating_listing", "excerpt_listing"]]
df["source"] = "AirlineRatings"

df["review_date"] = pd.to_datetime(
    df["review_date"],
    format="%d %B, %Y",
    errors="coerce"
)

df["rating_listing"] = df["rating_listing"].astype("float")
print("Missing/invalid dates:", df["review_date"].isna().sum())
df.rename(columns={"rating_listing":"rating", 'excerpt_listing': 'review'}, inplace=True)
print(df.info())


df.head()

In [ ]:
#Select specific Flight Report Columns
temp = df_fr[["airline", "publication_date", "airline_overall", "conclusion", "source"]]
temp.rename(columns={"publication_date":"review_date", "airline_overall": "rating", "conclusion":"review"}, inplace=True)
temp["review_date"] = pd.to_datetime(
    temp["review_date"],
    format="%Y-%m-%d",
    errors="coerce"
)

df = pd.concat([df, temp], ignore_index=True)
print("Missing/invalid dates:", df["review_date"].isna().sum())
print(df.info())

df.tail()


In [ ]:
#Select specific eSky Columns
temp = df_esky[["airline", "publication_date", "overall_rating", "review_text", "source"]]
temp.rename(columns={"publication_date":"review_date", "overall_rating": "rating", "review_text":"review"}, inplace=True)
temp["review_date"] = pd.to_datetime(
    temp["review_date"],
    format="%Y-%m-%d",
    errors="coerce"
)

df = pd.concat([df, temp], ignore_index=True)
print("Missing/invalid dates:", df["review_date"].isna().sum())
print(df.info())

df.tail()

In [ ]:
#Select specific eSky Columns
temp = df_kayak[["airline", "publication_date", "overall_rating", "review_text", "source"]]
temp.rename(columns={"publication_date":"review_date", "overall_rating": "rating", "review_text":"review"}, inplace=True)
temp["review_date"] = pd.to_datetime(
    temp["review_date"],
    format="%Y-%m",
    errors="coerce"
)

df = pd.concat([df, temp], ignore_index=True)
print("Missing/invalid dates:", df["review_date"].isna().sum())
print(df.info())

df.tail()

In [ ]:
df["airline"].unique()

In [ ]:
df["airline"] = df["airline"].replace(to_replace = ["aegean_airlines", "air_france", "bangkok_airways", "lufthansa", "porter_airlines", "turkish_airlines", "virgin_australia", "Aegean", "Bangkok Airways" ], 
                                      value =["Aegean Airlines", "Air France", "Bangkok Air", "Lufthansa", "Porter Airlines", "Turkish Airlines", "Virgin Australia", "Aegean Airlines", "Bangkok Air"])
df["airline"].unique()

In [ ]:

# Ensure review_date is datetime
df["review_date"] = pd.to_datetime(df["review_date"], errors="coerce")

# Filter reviews that are not NA and not empty
df_filtered = df[
    df["review"].notna() &
    df["review"].astype(str).str.strip().ne("")
].copy()

# Function to create summary tables
def create_review_summary(data):
    """Return or process create review summary."""
    return pd.crosstab(
        index=data["airline"],
        columns=data["source"],
        margins=True,
        margins_name="Total"
    )

# Table 1: All reviews with non-NA review text
table_all = create_review_summary(df_filtered)

# Table 2: Reviews from 2024 onwards
table_2024 = create_review_summary(
    df_filtered[df_filtered["review_date"] >= "2024-01-01"]
)

# Table 3: Reviews from 2026 onwards
table_2026 = create_review_summary(
    df_filtered[df_filtered["review_date"] >= "2026-01-01"]
)

# Display all three tables
print("ALL REVIEWS")
display(table_all)

print("REVIEWS FROM 2024 ONWARDS")
display(table_2024)

print("REVIEWS FROM 2026 ONWARDS")
display(table_2026)

In [ ]:
with pd.ExcelWriter(LEGACY_DIR / "airline_review_summary.xlsx") as writer:
    table_all.to_excel(writer, sheet_name="All Reviews")
    table_2024.to_excel(writer, sheet_name="From 2024")
    table_2026.to_excel(writer, sheet_name="From 2026")


In [ ]:
df.shape

In [ ]:
import pandas as pd
from lingua import Language, LanguageDetectorBuilder
from tqdm.auto import tqdm

tqdm.pandas()

# Initialize language detector
detector = LanguageDetectorBuilder.from_all_languages().build()

# Function to detect language
def detect_language(text):
    """Return or process detect language."""
    if pd.isna(text) or not str(text).strip():
        return None

    language = detector.detect_language_of(str(text))

    if language is None:
        return "Unknown"

    return language.name.title()

# Apply to DataFrame
df["language"] = df["review"].progress_apply(detect_language)

In [ ]:
language_summary = (
    df["language"]
    .value_counts(dropna=False)
    .reset_index()
)

language_summary.columns = ["language", "reviews"]

language_summary["percentage"] = (
    language_summary["reviews"] / len(df) * 100
).round(2)

display(language_summary)

In [ ]:
df_to_translate = df[
    df["language"].notna() &
    (df["language"] != "English")
].copy()

print(f"Total reviews: {len(df):,}")
print(f"Reviews needing translation: {len(df_to_translate):,}")
print(f"English reviews: {(df['language'] == 'English').sum():,}")

In [ ]:
def detect_language_with_confidence(text):

    """Return or process detect language with confidence."""
    if pd.isna(text) or not str(text).strip():
        return pd.Series([None, None])

    confidence_values = detector.compute_language_confidence_values(
        str(text)
    )

    if not confidence_values:
        return pd.Series(["Unknown", 0.0])

    best = confidence_values[0]

    return pd.Series([
        best.language.name.title(),
        best.value
    ])

df[["language", "language_confidence"]] = (
    df["review"].progress_apply(detect_language_with_confidence)
)

In [ ]:
df_low_confidence = df[
    df["language_confidence"] < 0.80
]

display(
    df_low_confidence[
        ["airline", "review", "language", "language_confidence"]
    ].head(20)
)

In [ ]:
language_by_airline = pd.crosstab(
    df["airline"],
    df["language"],
    margins=True
)

display(language_by_airline)

In [ ]:
# Only non-English reviews
df_translate = df[
    df["language"].notna() &
    (df["language"] != "English")
].copy()

# Approximate word count
df_translate["word_count"] = (
    df_translate["review"].fillna("").str.split().str.len()
)

n = len(df_translate)
avg_words = df_translate["word_count"].mean()

# Estimate tokens
input_tokens = n * 100 + df_translate["word_count"].sum() * 1.3
output_tokens = df_translate["word_count"].sum() * 1.3

# GPT-4.1-mini pricing
cost = (
    input_tokens / 1_000_000 * 0.40 +
    output_tokens / 1_000_000 * 1.60
)

print(f"Reviews to translate: {n:,}")
print(f"Average words per review: {avg_words:.1f}")
print(f"Estimated API cost: ${cost:.2f}")

In [ ]:
from openai import OpenAI
from getpass import getpass

api_key = getpass("Enter your OpenAI API key: ")

client = OpenAI(api_key=api_key)

In [ ]:
response = client.responses.create(
    model="gpt-4.1-mini",
    input="Translate into English: Η πτήση ήταν πολύ καλή."
)

print(response.output_text)

In [ ]:
print("Input tokens:", response.usage.input_tokens)
print("Output tokens:", response.usage.output_tokens)
print("Total tokens:", response.usage.total_tokens)

In [ ]:

import pandas as pd
import numpy as np
import json
import time
from pathlib import Path
from tqdm.auto import tqdm
from openai import OpenAI
from lingua import LanguageDetectorBuilder

# ============================================================
# 1. CONFIGURATION
# ============================================================

MODEL = "gpt-4.1-mini"

CHECKPOINT_FILE = (LEGACY_DIR / "airline_translation_checkpoint.pkl")
OUTPUT_FILE = (LEGACY_DIR / "airline_reviews_translated.csv")

SAVE_EVERY = 25

# API pricing per 1 million tokens (USD)
INPUT_PRICE = 0.40
OUTPUT_PRICE = 1.60

# Assumes 'client' has already been initialized with your API key
# client = OpenAI(api_key=api_key)

# ============================================================
# 2. PREPARE DATAFRAME
# ============================================================

df_final = df.copy().reset_index(drop=True)

# Detect language locally if not already available
if "language" not in df_final.columns:

    detector = LanguageDetectorBuilder.from_all_languages().build()

    def detect_language(text):
        """Return or process detect language."""
        if pd.isna(text) or not str(text).strip():
            return None

        lang = detector.detect_language_of(str(text))

        return lang.name.title() if lang else "Unknown"

    tqdm.pandas(desc="Detecting languages")

    df_final["language"] = (
        df_final["review"].progress_apply(detect_language)
    )

df_final["original_language"] = df_final["language"]

# Create output columns
df_final["review_en"] = df_final["review"].copy()
df_final["translated"] = False
df_final["translation_status"] = "pending"

# Mark English and missing reviews
english_mask = (
    df_final["original_language"].eq("English")
    & df_final["review"].notna()
)

empty_mask = (
    df_final["review"].isna()
    | df_final["review"].astype(str).str.strip().eq("")
)

df_final.loc[english_mask, "translation_status"] = "original_english"
df_final.loc[empty_mask, "translation_status"] = "empty"
df_final.loc[empty_mask, "review_en"] = None

# ============================================================
# 3. RESUME FROM CHECKPOINT
# ============================================================

if CHECKPOINT_FILE.exists():

    saved = pd.read_pickle(CHECKPOINT_FILE)

    if (
        len(saved) != len(df_final)
        or not saved["review"].fillna("").equals(
            df_final["review"].fillna("")
        )
    ):
        raise ValueError(
            "Checkpoint does not match the current DataFrame. "
            "Use a different checkpoint filename."
        )

    df_final = saved

    print("Previous checkpoint loaded.")

# ============================================================
# 4. TRANSLATION FUNCTION
# ============================================================

def translate_review(text, language):

    """Return or process translate review."""
    response = client.responses.create(
        model=MODEL,

        instructions=f"""
        You are a professional airline review translator.

        The original language is believed to be: {language}.

        Translate the following passenger review into English.

        Requirements:
        - Preserve the complete meaning.
        - Preserve sentiment, tone, sarcasm and criticism.
        - Do not summarize.
        - Do not add information.
        - Preserve airline names, airport codes and numbers.
        - If the review is already in English, return it unchanged.
        - Return only the English review text.
        """,

        input=str(text),

        max_output_tokens=2000
    )

    return (
        response.output_text.strip(),
        response.usage.input_tokens,
        response.usage.output_tokens
    )

# ============================================================
# 5. TRANSLATE NON-ENGLISH REVIEWS
# ============================================================

mask = (
    df_final["review"].notna()
    & ~df_final["original_language"].eq("English")
    & ~df_final["translation_status"].eq("completed")
    & ~df_final["translation_status"].eq("empty")
)

pending_indices = df_final.index[mask]

print(f"Total reviews: {len(df_final):,}")
print(f"Reviews requiring translation: {len(pending_indices):,}")

total_input_tokens = 0
total_output_tokens = 0
successful = 0
failed = 0

# Reuse translations for identical review texts
cache = {}

for _, row in df_final[
    df_final["translation_status"] == "completed"
].iterrows():

    if pd.notna(row["review_en"]):
        cache[str(row["review"])] = row["review_en"]

for count, idx in enumerate(
    tqdm(pending_indices, desc="Translating reviews"),
    start=1
):

    text = df_final.at[idx, "review"]
    language = df_final.at[idx, "original_language"]

    try:

        if str(text) in cache:
            translated_text = cache[str(text)]
            input_tokens = 0
            output_tokens = 0

        else:
            translated_text, input_tokens, output_tokens = (
                translate_review(text, language)
            )

            cache[str(text)] = translated_text

        df_final.at[idx, "review_en"] = translated_text
        df_final.at[idx, "translated"] = True
        df_final.at[idx, "translation_status"] = "completed"

        total_input_tokens += input_tokens
        total_output_tokens += output_tokens

        successful += 1

    except Exception as e:

        failed += 1

        df_final.at[idx, "translation_status"] = "error"

        print(f"\nError at row {idx}: {type(e).__name__}: {e}")

    # Save progress
    if count % SAVE_EVERY == 0:

        df_final.to_pickle(CHECKPOINT_FILE)

        current_cost = (
            total_input_tokens / 1_000_000 * INPUT_PRICE
            + total_output_tokens / 1_000_000 * OUTPUT_PRICE
        )

        print(
            f"Processed: {count:,}/{len(pending_indices):,} | "
            f"Successful: {successful:,} | "
            f"Failed: {failed:,} | "
            f"Session cost: ${current_cost:.4f}"
        )

# ============================================================
# 6. SAVE FINAL DATA
# ============================================================

df_final.to_pickle(CHECKPOINT_FILE)
df_final.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

total_cost = (
    total_input_tokens / 1_000_000 * INPUT_PRICE
    + total_output_tokens / 1_000_000 * OUTPUT_PRICE
)

print("\nTRANSLATION COMPLETED")
print("=" * 50)
print(f"Successfully processed: {successful:,}")
print(f"Failed: {failed:,}")
print(f"Input tokens: {total_input_tokens:,}")
print(f"Output tokens: {total_output_tokens:,}")
print(f"Estimated session cost: ${total_cost:.4f}")
print(f"Saved to: {OUTPUT_FILE}")

# ============================================================
# 7. PREVIEW RESULTS
# ============================================================

display(
    df_final[
        [
            "airline",
            "review_date",
            "original_language",
            "review",
            "review_en",
            "translated",
            "translation_status"
        ]
    ].head(20)
)


In [ ]:
df_final["translation_status"].value_counts()

In [ ]:
empty_reviews = df_final[
    df_final["translation_status"] == "empty"
].copy()

print("Total empty records:", len(empty_reviews))

print(
    "Records with non-empty original review:",
    empty_reviews["review"].notna().sum()
    - empty_reviews["review"].fillna("").astype(str).str.strip().eq("").sum()
    + empty_reviews["review"].isna().sum()
)

display(
    empty_reviews[
        ["airline", "review", "original_language", "review_en"]
    ].head(20)
)

In [ ]:
df_final.head(12)

In [ ]:
df.head(12)

In [ ]:
df_ar.head(12)

In [ ]:

import re
import math
import numpy as np
import pandas as pd
import airportsdata
from tqdm.auto import tqdm

tqdm.pandas()

# ============================================================
# 1. CONFIGURATION
# ============================================================

LONG_HAUL_THRESHOLD = 6.0

# Approximate effective speed and ground/terminal overhead
CRUISE_SPEED_KMH = 800
FLIGHT_OVERHEAD_HOURS = 0.5

# Flag estimates near the threshold rather than forcing a label
BORDERLINE_MARGIN = 0.5

airports = airportsdata.load("IATA")

# ============================================================
# 2. AIRPORT DISTANCE AND ESTIMATED DURATION
# ============================================================

def haversine_distance(lat1, lon1, lat2, lon2):

    """Return or process haversine distance."""
    R = 6371.0

    lat1, lon1, lat2, lon2 = map(
        math.radians, [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1) * math.cos(lat2)
        * math.sin(dlon / 2) ** 2
    )

    return 2 * R * math.asin(math.sqrt(a))


def get_flight_estimate(origin, destination):

    """Return or process get flight estimate."""
    if pd.isna(origin) or pd.isna(destination):
        return None, None

    origin = str(origin).strip().upper()
    destination = str(destination).strip().upper()

    if origin not in airports or destination not in airports:
        return None, None

    if origin == destination:
        return None, None

    a = airports[origin]
    b = airports[destination]

    distance = haversine_distance(
        a["lat"], a["lon"],
        b["lat"], b["lon"]
    )

    estimated_hours = (
        distance / CRUISE_SPEED_KMH
        + FLIGHT_OVERHEAD_HOURS
    )

    return round(distance, 1), round(estimated_hours, 2)


# ============================================================
# 3. EXTRACT ROUTES FROM STRUCTURED DATA
# ============================================================

def clean_iata(value):

    """Return or process clean iata."""
    if pd.isna(value):
        return None

    value = str(value).strip().upper()

    return value if re.fullmatch(r"[A-Z]{3}", value) else None


def extract_structured_route(row):

    """Return or process extract structured route."""
    origin = clean_iata(row.get("origin"))
    destination = clean_iata(row.get("destination"))

    if origin and destination:
        return origin, destination

    route = row.get("route")

    if pd.notna(route):

        match = re.search(
            r"\b([A-Z]{3})\s*[-–—/]\s*([A-Z]{3})\b",
            str(route).upper()
        )

        if match:
            return match.group(1), match.group(2)

    return None, None


# ============================================================
# 4. EXTRACT ROUTES FROM REVIEW TEXT
# ============================================================

ROUTE_PATTERNS = [
    r"\b([A-Z]{3})\s*(?:-|–|—|→|to)\s*([A-Z]{3})\b",
    r"\bfrom\s+([A-Z]{3})\s+to\s+([A-Z]{3})\b"
]


def extract_text_routes(text):

    """Return or process extract text routes."""
    if pd.isna(text):
        return []

    text = str(text)
    routes = []

    for pattern in ROUTE_PATTERNS:

        for origin, destination in re.findall(
            pattern, text, flags=re.IGNORECASE
        ):

            origin = origin.upper()
            destination = destination.upper()

            if (
                origin in airports
                and destination in airports
                and origin != destination
            ):
                route = (origin, destination)

                if route not in routes:
                    routes.append(route)

    return routes


# ============================================================
# 5. EXTRACT EXPLICIT FLIGHT DURATION
# ============================================================

DURATION_PATTERNS = [
    # 7-hour flight, 7 hour flight, 7.5-hour journey
    r"\b(\d+(?:[.,]\d+)?)\s*[- ]?\s*"
    r"(?:hours?|hrs?)\s*"
    r"(?:long\s+)?(?:flight|journey)\b",

    # flight lasted 7 hours, flight of 7 hours
    r"\b(?:flight|journey)\s+"
    r"(?:lasted|took|of|was)\s+"
    r"(\d+(?:[.,]\d+)?)\s*(?:hours?|hrs?)\b",

    # 7h 30m flight
    r"\b(\d{1,2})\s*h\s*(\d{1,2})\s*m?"
    r"\s*(?:flight|journey)\b"
]


def extract_flight_duration(text):

    """Return or process extract flight duration."""
    if pd.isna(text):
        return None

    text = str(text).lower()

    for i, pattern in enumerate(DURATION_PATTERNS):

        match = re.search(pattern, text)

        if match:

            if i == 2:
                hours = float(match.group(1))
                minutes = float(match.group(2))

                if minutes >= 60:
                    continue

                duration = hours + minutes / 60

            else:
                duration = float(
                    match.group(1).replace(",", ".")
                )

            if 0.25 <= duration <= 24:
                return round(duration, 2)

    return None


# ============================================================
# 6. DETECT LONG-HAUL LANGUAGE
# ============================================================

LONG_HAUL_PATTERN = re.compile(
    r"\b(?:long[- ]haul|long[- ]distance)\s+"
    r"(?:flight|route|service|journey)\b",
    re.IGNORECASE
)

SHORT_HAUL_PATTERN = re.compile(
    r"\b(?:short[- ]haul|short[- ]distance)\s+"
    r"(?:flight|route|service|journey)\b",
    re.IGNORECASE
)


def detect_haul_mention(text):

    """Return or process detect haul mention."""
    if pd.isna(text):
        return None

    text = str(text)

    long_mention = bool(LONG_HAUL_PATTERN.search(text))
    short_mention = bool(SHORT_HAUL_PATTERN.search(text))

    if long_mention and not short_mention:
        return "long_haul"

    if short_mention and not long_mention:
        return "short_haul"

    return None


# ============================================================
# 7. CLASSIFICATION FUNCTION
# ============================================================

def classify_haul(row):

    """Return or process classify haul."""
    text = row.get("review_en")

    if pd.isna(text) or not str(text).strip():
        text = row.get("review")

    result = {
        "haul_type": "Unknown",
        "haul_method": "insufficient_information",
        "haul_confidence": "unknown",
        "haul_origin": None,
        "haul_destination": None,
        "distance_km": np.nan,
        "estimated_duration_hours": np.nan,
        "explicit_duration_hours": np.nan
    }

    # Priority 1: Structured origin/destination
    origin, destination = extract_structured_route(row)

    route_method = "structured_route"

    # Priority 2: Route extracted from review text
    if not origin or not destination:

        routes = extract_text_routes(text)

        if len(routes) == 1:
            origin, destination = routes[0]
            route_method = "text_route"

    # Calculate distance and duration estimate
    if origin and destination:

        distance, hours = get_flight_estimate(
            origin, destination
        )

        if hours is not None:

            result["haul_origin"] = origin
            result["haul_destination"] = destination
            result["distance_km"] = distance
            result["estimated_duration_hours"] = hours
            result["haul_method"] = route_method

            if hours >= LONG_HAUL_THRESHOLD + BORDERLINE_MARGIN:
                result["haul_type"] = "Long-haul"
                result["haul_confidence"] = "estimated"

            elif hours < LONG_HAUL_THRESHOLD - BORDERLINE_MARGIN:
                result["haul_type"] = "Short/medium-haul"
                result["haul_confidence"] = "estimated"

            else:
                result["haul_type"] = "Unknown"
                result["haul_confidence"] = "borderline"

            return pd.Series(result)

    # Priority 3: Explicit flight duration in review
    duration = extract_flight_duration(text)

    if duration is not None:

        result["explicit_duration_hours"] = duration

        result["haul_type"] = (
            "Long-haul"
            if duration >= LONG_HAUL_THRESHOLD
            else "Short/medium-haul"
        )

        result["haul_method"] = "explicit_duration"
        result["haul_confidence"] = "text_evidence"

        return pd.Series(result)

    # Priority 4: Explicit long/short-haul mention
    mention = detect_haul_mention(text)

    if mention == "long_haul":

        result["haul_type"] = "Long-haul"
        result["haul_method"] = "text_long_haul_mention"
        result["haul_confidence"] = "text_evidence"

    elif mention == "short_haul":

        result["haul_type"] = "Short/medium-haul"
        result["haul_method"] = "text_short_haul_mention"
        result["haul_confidence"] = "text_evidence"

    return pd.Series(result)


# ============================================================
# 8. APPLY TO DATAFRAME
# ============================================================

haul_results = df_final.progress_apply(
    classify_haul,
    axis=1
)

df_final = pd.concat(
    [
        df_final.drop(
            columns=haul_results.columns,
            errors="ignore"
        ),
        haul_results
    ],
    axis=1
)

print("\nHAUL CLASSIFICATION RESULTS")
print("=" * 50)

print(df_final["haul_type"].value_counts(dropna=False))

print("\nClassification methods:")
print(df_final["haul_method"].value_counts())

# ============================================================
# 9. SAVE
# ============================================================

df_final.to_csv(
    LEGACY_DIR / "airline_reviews_with_haul.csv",
    index=False,
    encoding="utf-8-sig"
)

display(
    df_final[
        [
            "airline",
            "source",
            "haul_type",
            "haul_method",
            "haul_origin",
            "haul_destination",
            "estimated_duration_hours"
        ]
    ].head(20)
)


In [ ]:
haul_summary = pd.crosstab(
    df_final["airline"],
    df_final["haul_type"],
    margins=True,
    margins_name="Total"
)

display(haul_summary)

In [ ]:
df_final["long_haul"] = (df_final["haul_type"] == "Long-haul").astype(int)

In [ ]:
df_final.head()

In [ ]:
df_final.to_csv(LEGACY_DIR / "df_final.csv")